# 모두몰 조회 쿼리 모음

모두몰의 고객(`customers`)·주문(`orders`) 데이터로 **비즈니스 질문 5개**를 만들고 SQL로 답합니다.

**과제 조건 충족 여부 (Q1~Q5 기준)**

| 조건 | 사용한 곳 |
|---|---|
| `AND` | Q1, Q2, Q3, Q4, Q5 |
| `OR` | Q1, Q5 |
| `IN` / `NOT IN` | Q1, Q2, Q3, Q4, Q5 |
| `BETWEEN` | Q4 |
| 비교 연산자 (`>=`, `<>`) | Q1, Q3 |
| `IS NULL` | Q1, Q5 |

## 0. 환경 준비
### 0-1. 패키지 설치

In [25]:
%pip install duckdb pandas -q

Note: you may need to restart the kernel to use updated packages.


### 0-2. DuckDB 연결
메모리에 임시 데이터베이스를 만들고, SQL 결과를 표로 보여 주는 함수 `q()`를 정의합니다.

In [26]:
import duckdb

con = duckdb.connect()   # 메모리 DB

def q(sql):
    """SQL을 실행하고 결과를 표(DataFrame)로 반환"""
    return con.sql(sql).df()

### 0-3. 테이블 만들기

In [27]:
con.execute("""
CREATE OR REPLACE TABLE customers (
    customer_id VARCHAR,
    name        VARCHAR,
    country     VARCHAR,
    signup_date DATE,
    grade       VARCHAR
)
""")

con.execute("""
CREATE OR REPLACE TABLE orders (
    order_id    VARCHAR,
    customer_id VARCHAR,
    order_date  DATE,
    status      VARCHAR,
    amount      INTEGER
)
""")
print("테이블 생성 완료")

테이블 생성 완료


### 0-4. 데이터 넣기

In [28]:
con.execute("""
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA',   '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA',   '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL,    '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA',   '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')
""")

con.execute("""
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid',      125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped',    89000),
  ('O0003', 'C001', '2023-09-11', 'Returned',   45000),
  ('O0004', 'C003', '2023-09-15', 'Paid',      230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled',   NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped',    67000),
  ('O0007', 'C002', '2023-10-01', 'Paid',      158000),
  ('O0008', 'C006', '2023-10-04', 'Placed',     32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped',   410000),
  ('O0010', 'C008', '2023-10-19', 'Paid',       99000),
  ('O0011', 'C001', '2023-10-23', 'Paid',       76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled',   NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped',   142000),
  ('O0014', 'C004', '2023-11-08', 'Paid',       88000),
  ('O0015', 'C011', '2023-11-13', 'Placed',     53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped',   175000),
  ('O0017', 'C002', '2023-11-24', 'Returned',   61000),
  ('O0018', 'C013', '2023-11-29', 'Paid',      320000),
  ('O0019', 'C005', '2023-12-03', 'Paid',       47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped',   215000),
  ('O0021', 'C014', '2023-12-14', 'Placed',     38000),
  ('O0022', 'C001', '2023-12-20', 'Paid',      134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped',    92000),
  ('O0024', 'C007', '2024-01-03', 'Paid',      268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled',   NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid',      119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped',   405000),
  ('O0028', 'C006', '2024-01-28', 'Paid',       58000),
  ('O0029', 'C004', '2024-02-04', 'Returned',   73000),
  ('O0030', 'C012', '2024-02-11', 'Paid',      187000)
""")
print("데이터 적재 완료")

데이터 적재 완료


### 0-5. 적재 확인
고객 15명, 주문 30건이 들어갔는지 확인합니다.

In [29]:
q("""
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
""")

,table_name,row_count
0,customers,15
1,orders,30


In [30]:
q('DESCRIBE customers')

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,signup_date,DATE,YES,None,None,None
4,grade,VARCHAR,YES,None,None,None


In [31]:
q('DESCRIBE orders')

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_date,DATE,YES,None,None,None
3,status,VARCHAR,YES,None,None,None
4,amount,INTEGER,YES,None,None,None


> 매출로 인정하는 주문은 결제 완료(`Paid`)와 배송 중(`Shipped`)으로 정의합니다.
> `Placed`(접수만 됨), `Cancelled`(취소), `Returned`(반품)는 매출에서 제외합니다.

---
# 질문 (Q1 ~ Q5)

## Q1. 등급 상향 후보: Gold가 아닌데 20만 원 이상 고액 구매를 한 고객은?
사용 연산자: `IN`, `AND`, `>=`, `<>`, `OR`, `IS NULL` · 서브쿼리

In [32]:
q("""
SELECT *
FROM orders
WHERE status IN ('Paid', 'Shipped')
  AND amount >= 200000
  AND customer_id IN (
        SELECT customer_id
        FROM customers
        WHERE grade <> 'Gold'
           OR grade IS NULL      -- 등급 미지정 고객도 포함
      )
ORDER BY amount DESC
""")

,order_id,customer_id,order_date,status,amount
0,O0009,C007,2023-10-12,Shipped,410000
1,O0024,C007,2024-01-03,Paid,268000
2,O0004,C003,2023-09-15,Paid,230000


**💡 발견:** Silver인 **C007**이 41만 원·26.8만 원 두 차례 고액 구매(합계 67.8만 원)를 해 Gold 이상의 실적을 보이므로 등급 상향 1순위 후보다. `grade <> 'Gold'`만 쓰면 NULL 등급 고객이 빠지므로 `OR grade IS NULL`을 함께 써야 한다.

## Q2. Gold 등급 리스크: Gold 고객 중 취소·반품을 낸 주문은?
사용 연산자: `IN`, `AND` · 서브쿼리

In [33]:
q("""
SELECT *
FROM orders
WHERE status IN ('Cancelled', 'Returned')
  AND customer_id IN (
        SELECT customer_id
        FROM customers
        WHERE grade = 'Gold'
      )
ORDER BY customer_id, order_date
""")

,order_id,customer_id,order_date,status,amount
0,O0003,C001,2023-09-11,Returned,45000
1,O0005,C004,2023-09-20,Cancelled,<NA>
2,O0029,C004,2024-02-04,Returned,73000


**💡 발견:** Gold **C004**는 주문 3건 중 2건이 취소·반품이고 정상 구매는 8.8만 원 1건뿐이라, 현재 등급이 실제 기여도와 맞지 않아 등급 재심사 대상이다.

## Q3. 이탈 위험 고객: 2024년 들어 정상 구매가 한 번도 없는 고객은?
사용 연산자: `NOT IN`, `>=`, `AND`, `IN` · 서브쿼리

In [34]:
q("""
SELECT *
FROM customers
WHERE customer_id NOT IN (
        SELECT customer_id
        FROM orders
        WHERE order_date >= '2024-01-01'
          AND status IN ('Paid', 'Shipped')
      )
ORDER BY grade, customer_id
""")

,customer_id,name,country,signup_date,grade
0,C009,임하은,Korea,2023-05-22,Bronze
1,C011,오시우,None,2023-06-19,Bronze
2,C015,안수아,Korea,2023-08-28,Bronze
3,C001,김민준,Korea,2023-01-05,Gold
4,C004,최지우,USA,2023-03-03,Gold
5,C008,윤지호,USA,2023-05-09,Gold
6,C002,이서연,Korea,2023-02-11,Silver
7,C005,정하준,Korea,2023-03-15,Silver
8,C010,한예준,Korea,2023-06-02,Silver
9,C014,황지안,USA,2023-08-10,None


**💡 발견:** 15명 중 10명이 해당되고, 특히 Gold 4명 중 3명(C001·C004·C008)이 2024년에 정상 구매가 없어 핵심 고객 대상 재구매 캠페인이 시급하다. (2024년 데이터는 1~2월분만 있다는 점은 감안해야 한다.)

## Q4. 해외 시장 성과: 2023년 4분기에 일본·미국 고객이 낸 매출 주문은?
사용 연산자: `BETWEEN`, `IN`, `AND` · 서브쿼리

In [35]:
q("""
SELECT *
FROM orders
WHERE order_date BETWEEN '2023-10-01' AND '2023-12-31'
  AND status IN ('Paid', 'Shipped')
  AND customer_id IN (
        SELECT customer_id
        FROM customers
        WHERE country IN ('Japan', 'USA')
      )
ORDER BY amount DESC
""")

,order_id,customer_id,order_date,status,amount
0,O0009,C007,2023-10-12,Shipped,410000
1,O0020,C008,2023-12-09,Shipped,215000
2,O0016,C012,2023-11-19,Shipped,175000
3,O0010,C008,2023-10-19,Paid,99000
4,O0014,C004,2023-11-08,Paid,88000


**💡 발견:** 해외 매출 주문 5건의 합계는 98.7만 원으로, 같은 조건의 한국 고객 주문 7건 합계(96.9만 원)보다 많다. 고객 수는 한국(8명)이 해외(6명)보다 많으므로 해외 고객의 1인당 기여도가 더 높다.

## Q5. 데이터 품질 점검: 국가·등급 정보가 빠진 고객의 주문은 어떤 상태인가?
사용 연산자: `AND`, `IN`, `OR`, `IS NULL` · 서브쿼리

In [36]:
q("""
SELECT *
FROM orders
WHERE status = 'Placed'
  AND customer_id IN (
        SELECT customer_id
        FROM customers
        WHERE country IS NULL
           OR grade IS NULL
      )
""")

,order_id,customer_id,order_date,status,amount
0,O0015,C011,2023-11-13,Placed,53000
1,O0021,C014,2023-12-14,Placed,38000


**💡 발견:** 정보 누락 고객 2명(C011·C014)의 주문이 모두 `Placed`에서 멈춰 있고, 전체 미처리 주문 3건 중 2건이 여기에 해당한다. 배송지(국가)나 등급 정보가 없어 처리가 막혔을 가능성이 있으므로 고객 정보 보완이 먼저다.